# KiteFS Feature Store - Getting Started Showcase

This notebook demonstrates how to use KiteFS.

- **Phase 1**  Define feature groups, ingest prepared data, explore the local store, and train an initial model — entirely on your machine, no cloud setup needed. 
- **Phase 2**  Publish the feature store to AWS (S3 + DynamoDB) and materialize online features for serving. 
- **Phase 3**  One month later: ingest new data, refresh the online store, build a rolling training dataset, and retrain the model. 

**Scenario:** A Turkish real-estate platform wants to recommend a realistic sale price for new listings. The data set is containing the data of year 2025. Two feature groups capture what KiteFS stores and serves:

| Feature Group | Storage | Entity Key | Event Timestamp | Purpose |
|---|---|---|---|---|
| `listing_features` | Offline only | `listing_id` | `sold_at` | Historical sold listings used for training |
| `town_market_features` | Offline + Online | `town_id` | `event_timestamp` | Monthly town-level price aggregates for training and serving |

In [ ]:
from kitefs import FeatureStore
import pandas as pd

## Phase 1 - Local Development & Training

Start with the local runtime target (`KITEFS_RUNTIME_TARGET=local`, which is the default). KiteFS writes everything under `feature_store/` — no AWS credentials or cloud access needed at this stage.

### 1.1 Create a local FeatureStore

`FeatureStore()` reads `./kitefs.yaml`, resolves the runtime target, and builds the storage provider for that target. The instance is bound to its target for its lifetime; switching targets requires creating a new instance.


### Optional error-case demos

The notebook still runs the happy path with clean datasets. Error-case cells below are intentionally commented out; uncomment a block when you want to show how KiteFS responds to malformed inputs or invalid parameters.

The convention is to create in-memory copies named `malformed_*`, mutate one thing, and pass that copy to KiteFS. The clean datasets remain unchanged.

In [ ]:
store = FeatureStore()

### 1.2 Apply & inspect the registry

`apply()` discovers every `FeatureGroup` in `feature_store/definitions/*.py`, validates cross-definition rules (e.g. join key references), and writes `feature_store/registry.json`. The registry is the single source of truth for all downstream operations. Run `apply()` again whenever you change a definition.


In [ ]:
result = store.apply()
print(result)

In [ ]:
store.list_feature_groups()

**CLI alternative** — `kitefs apply`, `kitefs list`, and `kitefs describe` do the same from the terminal. The cells below run those commands directly in the notebook using `!` shell syntax.


In [ ]:
!kitefs list --format json

In [ ]:
!kitefs describe listing_features --format json

### 1.3 Prepare & ingest data

KiteFS does not compute features — your data pipeline does. `ingest()` accepts two input styles: a `pandas.DataFrame` or a file path (`.parquet` or `.csv`). Either way, the data goes through the same schema validation and is written to the offline store as partitioned Parquet under `feature_store/data/offline_store/<group>/year=YYYY/month=MM/`.

In [ ]:
listing_features_df = pd.read_parquet("data/listing_features.parquet")

In [ ]:
listing_features_df.head()

In [ ]:
# using ingest with dataframe
store.ingest("listing_features", listing_features_df)

# using ingest with file path
result = store.ingest("town_market_features", "data/town_market_features.parquet")
print(result)

**CLI alternative** — `kitefs ingest` accepts file paths only:
```bash
kitefs ingest listing_features data/listing_features.parquet
kitefs ingest town_market_features data/town_market_features.parquet
```


#### Optional error case: ingestion validation

Ingestion always checks that the incoming data matches the registered feature group: required columns must exist, structural columns must be non-null, and column types must align. After that, feature-level `Expect` rules run according to the group's validation mode.

`listing_features` uses `ERROR`, so a schema mismatch or invalid feature value rejects the batch and writes no rows.

In [ ]:
# Schema check: required column does not match the feature group.
# Uncomment to see KiteFS reject the batch before writing anything.
#
# malformed_listing_features_df = listing_features_df.rename(
#     columns={"listing_id": "listing_identifier"}
# )
# store.ingest("listing_features", malformed_listing_features_df)


# Structural validation: column type does not match the feature group.
# Uncomment to see KiteFS report that `net_area` is no longer an INTEGER column.
#
# malformed_listing_features_df = listing_features_df.copy()
# malformed_listing_features_df["net_area"] = malformed_listing_features_df["net_area"].astype(str)
# store.ingest("listing_features", malformed_listing_features_df)


# Value validation: feature values violate the Expect rules in ERROR mode.
# Uncomment to see the whole batch rejected and inspect the validation report.
#
# malformed_listing_features_df = listing_features_df.copy()
# malformed_listing_features_df.loc[0, "net_area"] = 0
# malformed_listing_features_df.loc[1, "build_year"] = 1899
# malformed_listing_features_df.loc[2, "sold_price"] = None
#
# try:
#     store.ingest("listing_features", malformed_listing_features_df)
# except Exception as err:
#     print(type(err).__name__)
#     print(err)
#     print(err.report)
#     print(err.report.failures[:3])


# Wrong feature group name: the registry is the contract.
# Uncomment to see KiteFS list the registered feature groups in the error message.
#
# store.ingest("listings", listing_features_df)

#### Validation modes: ERROR, FILTER, and NONE

The two demo feature groups are intentionally tiny and isolated from the real example data.

- The real groups use `ERROR`: any feature-rule failure rejects the whole ingest batch.
- `demo_filter_features` uses `FILTER`: invalid rows are dropped and valid rows are written.
- `demo_none_features` uses `NONE`: feature-level `Expect` rules are skipped, but structural checks still run.

Because these demo groups live in `feature_store/definitions/`, `store.apply()` registers them along with the real feature groups.

In [ ]:
demo_mode_df = pd.DataFrame(
    {
        "demo_id": [1, 2, 3],
        "event_timestamp": pd.to_datetime(
            ["2025-01-01", "2025-01-02", "2025-01-03"], utc=True
        ),
        "score": [10.0, -5.0, 20.0],
    }
)

filter_result = store.ingest("demo_filter_features", demo_mode_df)
print("FILTER accepted rows:", filter_result.accepted_rows)
print("FILTER rejected rows:", filter_result.rejected_rows)
print("FILTER validation report:", filter_result.validation_report)

none_result = store.ingest("demo_none_features", demo_mode_df)
print("NONE accepted rows:", none_result.accepted_rows)
print("NONE rejected rows:", none_result.rejected_rows)
print("NONE validation report:", none_result.validation_report)

### 1.4 Explore the offline store

`get_historical_features()` reads directly from the offline Parquet partitions. `select` names the feature columns to return. `where` filters by event timestamp. Structural columns (entity key, event timestamp, join keys) are always included in the result.


In [ ]:
from datetime import datetime

listings_wo_join = store.get_historical_features(
    from_="listing_features",
    select=[
        "net_area",
        "number_of_rooms",
        "build_year",
        "sold_price",
    ],
    where={
        "sold_at": {
            "gte": datetime(2025, 4, 1),
            "lte": datetime(2025, 4, 10, 23, 59, 59),
        }
    },
)

listings_wo_join

It is also possible to retrieve historical features by joining two feature groups. 

In [ ]:
listings = store.get_historical_features(
    from_="listing_features",
    join=["town_market_features"],
    select={
        "listing_features": [
            "net_area",
            "number_of_rooms",
            "build_year",
            "sold_price",
        ],
        "town_market_features": ["avg_price_per_sqm"],
    },
    where={
        "sold_at": {
            "gte": datetime(2025, 4, 1),
            "lte": datetime(2025, 4, 10, 23, 59, 59),
        }
    },
)

listings

### 1.5 Materialize & read from the online store

Materialization copies, for each entity key, the offline row with the **latest** `event_timestamp` into the online store. In local mode the online store is a SQLite database at `feature_store/data/online_store/online.db`.

Only groups with `StorageTarget.OFFLINE_AND_ONLINE` can be materialized. `listing_features` is offline-only — calling `materialize("listing_features")` would raise `FeatureGroupNotMaterializableError`.


In [ ]:
materialize_result = store.materialize("town_market_features")
print(materialize_result)

In [ ]:
# `listing_features` is offline-only, so it cannot be materialized to the online store.
# Uncomment to see KiteFS reject the request before doing any online-store work.
#
# store.materialize("listing_features")

Since in the local runtime the online store is sqlite, it is possible to connect it with any SQL client. And query on the database directly with SQL syntax.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("feature_store/data/online_store/online.db")

df = pd.read_sql_query("select * from town_market_features;", conn)
print(df)

conn.close()

Use get_online_features() to read from the online store. It works like get_historical_features with similar syntax, but always returns the latest features for each entity key, since the online store has the most recent data. 

In [ ]:
market_price = store.get_online_features(
    from_="town_market_features",
    select=["*"],
    where={"town_id": {"eq": 1}},
)

market_price

#### Optional error case: online lookup misses and parameters

An online lookup miss is not an error; KiteFS returns an empty dictionary so serving code can decide how to respond. Parameter errors are different: online lookup must filter the entity key with `eq`, and the lookup value must match the entity key type.

In [ ]:
# Uncomment either block to see KiteFS reject invalid online lookup parameters.
#
# store.get_online_features(
#     from_="town_market_features",
#     select=["avg_price_per_sqm"],
#     where={"town_id": {"gte": 1}},
# )
#
# store.get_online_features(
#     from_="town_market_features",
#     select=["avg_price_per_sqm"],
#     where={"avg_price_per_sqm": {"eq": 1000.0}},
# )

**CLI alternative** — `kitefs materialize [GROUP_NAME]` does the same from the terminal. Omit the group name to materialize all `OFFLINE_AND_ONLINE` groups at once:
```bash
kitefs materialize town_market_features
kitefs materialize  # all eligible groups
```

### 1.6 Build a training dataset (point-in-time join) & train the model

A **point-in-time join** anchors each training row at its event timestamp. For each listing row, KiteFS picks the most recent `town_market_features` row whose `event_timestamp` is **at or before** the listing's `sold_at`. This prevents future market data from leaking into training — the model only sees information that was actually available when each listing sold.

In [ ]:
from datetime import datetime
from pathlib import Path

import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

FEATURE_COLUMNS = [
    "net_area",
    "number_of_rooms",
    "build_year",
    "town_market_features_avg_price_per_sqm",
]
LABEL_COLUMN = "sold_price"


def build_training_dataset(store, *, until):
    return store.get_historical_features(
        from_="listing_features",
        join=["town_market_features"],
        select={
            "listing_features": [
                "net_area",
                "number_of_rooms",
                "build_year",
                "sold_price",
            ],
            "town_market_features": ["avg_price_per_sqm"],
        },
        where={
            "sold_at": {
                "gte": datetime(2025, 2, 1),
                "lte": until,
            }
        },
    )


def train_and_evaluate(training_df):
    X = training_df[FEATURE_COLUMNS]
    y = training_df[LABEL_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    print("MAE:", mean_absolute_error(y_test, predictions))
    print("R2 :", r2_score(y_test, predictions))

    return model


def save_model(model, path="models/model.pkl"):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, path)
    
    print(f"Model saved → {path}")


**Note:** In `build_training_dataset` method we are retrieving historical features starting from February 2025, not January 2025, because the point-in-time join needs to look back at least one month to get the first `town_market_features` record for January 2025. If we started from January 2025, those listings would have no matching market features and would be dropped from the training dataset. This is an architectural decision regarding to model training, not a KiteFS limitation.

In [ ]:
training_df = build_training_dataset(store, until=datetime(2025, 12, 31, 23, 59, 59))
training_df.head()


In [ ]:
model = train_and_evaluate(training_df)


In [ ]:
save_model(model)


## Phase 2 - Switch to Remote & Publish (AWS)

The local workflow is complete. To share features with the backend service and other team members, switch to the remote target.

**Before running these cells:**
1. Copy `.env.example` → `.env` in this directory.
2. Fill in your S3 bucket name, AWS profile, and region.
3. Confirm your AWS credentials are configured in `~/.aws/config`.

KiteFS uses the standard boto3 credential chain, it does not store or read credentials itself.

### 2.1 Load remote configuration


**Demo definition note:** This showcase includes two tiny validation-mode demo groups, so remote publishing will include them in the registry too. In a production feature store, remove `feature_store/definitions/demo_validation_modes.py` if you only want the real application groups.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env.
load_dotenv()

print("Runtime target:", os.environ.get("KITEFS_RUNTIME_TARGET", "(not set)"))
print("AWS region    :", os.environ.get("KITEFS_AWS_REGION", "(not set)"))


In [ ]:
# Build a fresh, AWS-bound store instance to resolve the environment variables and use them.
remote_store = FeatureStore()

As a side note, running `kitefs apply` will create the instance behind the scenes and load the configuration. So manual instantiation is needed only using SDK. 

### 2.2 Publish the registry and ingest data to S3

`apply(publish=True)` writes the registry to S3 in addition to the local file, making the feature group schemas available to any consumer project (e.g. the backend service). 

KiteFS does not create S3 buckets because of security and unique name requirements. So before moving to this phase, be sure to create the bucket you specified in `.env` and confirm your credentials have write access to it.


In [ ]:
apply_result = remote_store.apply(publish=True)
print("Registered groups:", apply_result.registered_groups)
print("Published successfully:", apply_result.published)

`publish=True` create the registry and uploads it to S3. When using the CLI:
```bash
kitefs apply --publish
```
will prompt you to confirm before proceeding. To bypass the prompt, add `--yes`:
```bash
kitefs apply --publish --yes
```
which is useful for CI/CD pipelines.

In [ ]:
print("Ingesting listing features to S3...")
remote_store.ingest("listing_features", "data/listing_features.parquet")

print("Ingesting town market aggregates to S3...")
remote_store.ingest("town_market_features", "data/town_market_features.parquet")

print("Ingestion complete!")

In [ ]:
remote_store.list_feature_groups()

### 2.3 Materialize to the remote online store (DynamoDB)

Materialization now writes to DynamoDB. Materialize will create the DynamoDB table if it doesn't exist, so no manual setup is needed.


In [ ]:
materialize_result = remote_store.materialize("town_market_features")
print(materialize_result)


## Phase 3 - Retraining Loop

Assuming one month later, January 2026 data is ready. The workflow is the same as Phase 1. This section uses the **CLI** as the primary surface to demonstrate it as a real alternative to the SDK and how it can be used during CI/CD pipelines. 

### 3.1 Ingest January 2026 data and refresh the online store

The CLI is the primary surface here. Each command is followed by its SDK equivalent in a callout block.


In [ ]:
!kitefs ingest listing_features data/listing_features_jan2026.parquet

In [ ]:
!kitefs ingest town_market_features data/town_market_jan2026.parquet

In [ ]:
!kitefs materialize town_market_features

In [ ]:
market_price = remote_store.get_online_features(
    from_="town_market_features",
    select=["*"],
    where={"town_id": {"eq": 1}},
)

print(market_price)

### 3.3 Rebuild the training dataset and retrain

Now we can move the rolling window forward by one month and retrain the model with the latest data.

In [ ]:
training_df = build_training_dataset(
    remote_store,
    until=datetime(2026, 1, 31, 23, 59, 59),
)


In [ ]:
training_df.describe()

In [ ]:
training_df.head()

In [ ]:
model = train_and_evaluate(training_df)
save_model(model)
